In [1]:
import zipfile
import pandas as pd
from pathlib import Path
import pyarrow.parquet as pq
import pyarrow as pa
from concurrent.futures import ThreadPoolExecutor
import tqdm
import traceback
import threading

In [2]:
# === 設定 ===
ZIP_DIR = Path("/Users/fang/Desktop/bs_report")         # 放 ZIP 的資料夾
OUTPUT_DIR = Path("/Users/fang/Desktop/bs_report/parquet") # 轉換後的輸出位置
OUTPUT_DIR.mkdir(exist_ok=True)
CSV_SUBPATH = "twse"                  # ZIP 內的子目錄

# ZIP_DIR = Path("zip_reports")
# OUTPUT_DIR = Path("parquet_by_stock")
# OUTPUT_DIR.mkdir(exist_ok=True)
# CSV_SUBPATH = "twse"

file_locks = {}
file_locks_lock = threading.Lock()

def get_lock(stock_id):
    with file_locks_lock:
        if stock_id not in file_locks:
            file_locks[stock_id] = threading.Lock()
        return file_locks[stock_id]

def safe_convert_numeric(series):
    return pd.to_numeric(series.str.replace(",", ""), errors="coerce")

def process_zip(zip_path: Path):
    date = zip_path.stem.split("_")[-1]
    with zipfile.ZipFile(zip_path, "r") as z:
        csv_files = [f for f in z.namelist() if f.startswith(CSV_SUBPATH) and f.endswith(".csv")]
        for csv_name in csv_files:
            stock_id = Path(csv_name).stem
            try:
                with z.open(csv_name) as f:
                    df = pd.read_csv(f, dtype=str).fillna("")
                    df["日期"] = pd.to_datetime(date)
                    # 統一數值欄型別
                    for col in ["價格", "買進股數", "賣出股數"]:
                        if col in df.columns:
                            import pandas as pd

# 讀取單一 parquet 檔案
df = pd.read_parquet("parquet_by_stock/2330.parquet")

print(df.head())
print(df.dtypes)df[col] = safe_convert_numeric(df[col]).astype("float64")
                    write_parquet_incremental(stock_id, df)
            except Exception as e:
                print(f"[WARN] {csv_name} in {zip_path.name} 讀取失敗: {e}")
                traceback.print_exc()
                continue

def write_parquet_incremental(stock_id, df):
    out_file = OUTPUT_DIR / f"{stock_id}.parquet"
    lock = get_lock(stock_id)
    with lock:
        try:
            new_table = pa.Table.from_pandas(df, preserve_index=False)
            if out_file.exists():
                old_table = pq.read_table(out_file)
                all_fields = sorted(set(old_table.schema.names) | set(new_table.schema.names))
                old_table = old_table.select([f for f in all_fields if f in old_table.schema.names])
                new_table = new_table.select([f for f in all_fields if f in new_table.schema.names])
                combined = pa.concat_tables(
                    [old_table, new_table],
                    promote_options="default"  # 取代舊版 promote=True
                )
                pq.write_table(combined, out_file, compression="zstd")
            else:
                pq.write_table(new_table, out_file, compression="zstd")

        except pa.ArrowTypeError as e:
            print(f"[WARN] {stock_id} schema mismatch, retrying as float64: {e}")
            # 若發生型別不一致，再轉 float64 重試
            for col in ["價格", "買進股數", "賣出股數"]:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")
            table = pa.Table.from_pandas(df, preserve_index=False)
            pq.write_table(table, out_file, compression="zstd")

        except Exception as e:
            print(f"[ERROR] 寫入 {stock_id}.parquet 出錯: {e}")
            traceback.print_exc()

# === 主程式 ===
zip_files = sorted(ZIP_DIR.glob("bs_report_*.zip"))
print(f"共找到 {len(zip_files)} 個 ZIP，開始並行處理...")

with ThreadPoolExecutor(max_workers=6) as executor:
    list(tqdm.tqdm(executor.map(process_zip, zip_files), total=len(zip_files)))

print("✅ 全部完成！")

共找到 99 個 ZIP，開始並行處理...


  0%|                                          | 0/99 [00:00<?, ?it/s]

[WARN] 1104 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 1109 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 1110 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 1203 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 1210 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 1213 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 1215 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 1216 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 1217 schema mismatch, retrying as float64: Unable

  0%|                                          | 0/99 [00:03<?, ?it/s]

[WARN] 1402 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 1413 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double


[WARN] 1416 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 1417 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 1418 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 1419 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 1423 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 1432 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 1435 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 1439 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 1440 schema mismatch, retrying as float64: Unable

KeyboardInterrupt: 

[WARN] 2363 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 2367 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 2368 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 2369 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 2371 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 2375 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 2377 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 2379 schema mismatch, retrying as float64: Unable to merge: Field 買進股數 has incompatible types: int64 vs double
[WARN] 2382 schema mismatch, retrying as float64: Unable

Traceback (most recent call last):
  File "/var/folders/mt/8vtyxhqn6fs1r_f4qbgypfwh0000gn/T/ipykernel_19948/1639641614.py", line 51, in write_parquet_incremental
    old_table = pq.read_table(out_file)
                ^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/fang/StockAnalysis/.venv/lib/python3.11/site-packages/pyarrow/parquet/core.py", line 1898, in read_table
    return dataset.read(columns=columns, use_threads=use_threads,
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/fang/StockAnalysis/.venv/lib/python3.11/site-packages/pyarrow/parquet/core.py", line 1538, in read
    table = self._dataset.to_table(
            ^^^^^^^^^^^^^^^^^^^^^^^
  File "pyarrow/_dataset.pyx", line 589, in pyarrow._dataset.Dataset.to_table
  File "pyarrow/_dataset.pyx", line 3939, in pyarrow._dataset.Scanner.to_table
  File "pyarrow/error.pxi", line 155, in pyarrow.lib.pyarrow_internal_check_status
  File "pyarrow/error.pxi", line 92, in pyarrow.lib.check_status
FileNotFoundErr

In [4]:
old_table

pyarrow.Table
券商: string
價格: double
買進股數: double
賣出股數: string
日期: string
date: string
----
券商: [["1020","1022","1033","1114","1440",...,"989f","989f","9A00","9A9h",null]]
價格: [[13.75,13.6,13.8,13.8,13.8,...,13.8,13.85,13.85,13.75,null]]
買進股數: [[1000,1000,1000,0,7000,...,0,0,0,3000,null]]
賣出股數: [["0","0 ","0","1000 ","0",...,"20000 ","13000","4000 ","0"," "]]
日期: [["2025/04/15","2025/04/15","2025/04/15","2025/04/15","2025/04/15",...,"2025/04/15","2025/04/15","2025/04/15","2025/04/15","2025/04/15"]]
date: [["20250415","20250415","20250415","20250415","20250415",...,"20250415","20250415","20250415","20250415","20250415"]]

In [5]:
table

pyarrow.Table
券商: string
價格: double
買進股數: int64
賣出股數: int64
日期: string
date: string
----
券商: [["1043","104A","1116","1119","1440",...,"9A00","9A00","9A00","9A9U","9A9U"]]
價格: [[13.8,13.75,13.85,13.8,13.8,...,13.8,13.85,13.9,13.75,13.9]]
買進股數: [[2000,1000,2000,0,3000,...,6000,0,0,19000,0]]
賣出股數: [[0,0,0,19000,1000,...,283,3,409,0,23000]]
日期: [["2025/04/16","2025/04/16","2025/04/16","2025/04/16","2025/04/16",...,"2025/04/16","2025/04/16","2025/04/16","2025/04/16","2025/04/16"]]
date: [["20250416","20250416","20250416","20250416","20250416",...,"20250416","20250416","20250416","20250416","20250416"]]

In [14]:
import pandas as pd

# 讀取單一 parquet 檔案
df = pd.read_parquet("/Users/fang/Desktop/bs_report/parquet_tpex/735905.parquet")

print(df.head())
print(df.dtypes)

     價格    券商         日期    買進股數    賣出股數
0  2.15  5920 2025-07-17  5000.0     0.0
1  2.15  7790 2025-07-17     0.0  5000.0
2  2.13  5920 2025-07-16  8000.0     0.0
3  2.18  5920 2025-07-16  3000.0     0.0
4  2.20  5920 2025-07-16     0.0  5000.0
價格             float64
券商              object
日期      datetime64[ns]
買進股數           float64
賣出股數           float64
dtype: object


In [15]:
df.groupby('日期').count()

,價格,券商,買進股數,賣出股數
日期,,,,
2025-07-16,12,12,12,12
2025-07-17,4,4,4,4
2025-07-21,8,8,8,8
2025-07-22,4,4,4,4
2025-07-23,8,8,8,8
2025-07-24,8,8,8,8
2025-08-12,12,12,12,12
2025-08-13,8,8,8,8
2025-08-14,4,4,4,4


In [16]:
from pathlib import Path


INPUT_DIR = Path("/Users/fang/Desktop/bs_report/bs_report/twse")           # 放每日資料夾的根目錄


unique_symbols = set()
total_files = 0

for day_path in sorted(INPUT_DIR.iterdir()):
    if not day_path.is_dir():
        continue

    for csv_path in day_path.glob("*.csv"):
        unique_symbols.add(csv_path.stem)
        total_files += 1

print(f"Total CSV files: {total_files}")
print(f"Unique stock symbols: {len(unique_symbols)}")


Total CSV files: 118539
Unique stock symbols: 23703


In [17]:
import zipfile
from pathlib import Path


ZIP_DIR = Path("/Users/fang/Desktop/bs_report/twse_zip")


def extract_symbols_from_zip(zip_path: Path):
    symbols = set()
    csv_count = 0

    with zipfile.ZipFile(zip_path, "r") as zf:
        for name in zf.namelist():
            if not name.endswith(".csv"):
                continue
            csv_count += 1
            symbols.add(Path(name).stem)

    return csv_count, symbols



totals = {}
grand_csv_count = 0
grand_unique_symbols = set()

for zip_path in sorted(ZIP_DIR.glob("*.zip")):
    csv_count, symbols = extract_symbols_from_zip(zip_path)
    totals[zip_path.name] = {
        "csv_count": csv_count,
        "unique_symbols": len(symbols),
    }
    grand_csv_count += csv_count
    grand_unique_symbols.update(symbols)

for name, data in totals.items():
    print(f"{name}: {data['csv_count']} CSV files, {data['unique_symbols']} unique symbols")

print("---")
print(f"Total CSV files across zips: {grand_csv_count}")
print(f"Unique stock symbols across zips: {len(grand_unique_symbols)}")


bs_report_20250415.zip: 10273 CSV files, 10273 unique symbols
bs_report_20250416.zip: 9499 CSV files, 9499 unique symbols
bs_report_20250417.zip: 9153 CSV files, 9153 unique symbols
bs_report_20250418.zip: 8753 CSV files, 8753 unique symbols
bs_report_20250421.zip: 9008 CSV files, 9008 unique symbols
bs_report_20250422.zip: 8994 CSV files, 8994 unique symbols
bs_report_20250423.zip: 10688 CSV files, 10688 unique symbols
bs_report_20250424.zip: 9632 CSV files, 9632 unique symbols
bs_report_20250425.zip: 9344 CSV files, 9344 unique symbols
bs_report_20250428.zip: 9535 CSV files, 9535 unique symbols
bs_report_20250429.zip: 10636 CSV files, 10636 unique symbols
bs_report_20250430.zip: 10518 CSV files, 10518 unique symbols
bs_report_20250502.zip: 12306 CSV files, 12306 unique symbols
bs_report_20250505.zip: 11943 CSV files, 11943 unique symbols
bs_report_20250506.zip: 10660 CSV files, 10660 unique symbols
bs_report_20250507.zip: 10464 CSV files, 10464 unique symbols
bs_report_20250508.zip: 

In [38]:
import pandas as pd
from pathlib import Path

# 1. 設定股票、期間與資料檔
stock = "1216"
start = "2025-06-15"
end = "2025-07-15"
parquet_path = Path("~/Desktop/bs_report/parquet_twse") / f"{stock}.parquet"  # 需依實際路徑調整

# 2. 載入資料
df = pd.read_parquet(parquet_path)

# 3. 基礎清理與篩選
df["日期"] = pd.to_datetime(df["日期"], errors="coerce")
df = df.loc[(df["日期"] >= start) & (df["日期"] <= end)].copy()

numeric_cols = ["買進股數", "賣出股數"]
for col in numeric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({"": "0", "nan": "0", "None": "0"})
        .astype(float)
    )

df["淨買超"] = df["買進股數"] - df["賣出股數"]

# 4. 依券商彙總
summary = (
    df.groupby("券商", as_index=False)
      .agg(
          買進股數=("買進股數", "sum"),
          賣出股數=("賣出股數", "sum"),
          淨買超=("淨買超", "sum"),
      )
)

total_buy = summary["買進股數"].sum()
total_net = summary["淨買超"].sum()

summary["買進占比"] = summary["買進股數"] / total_buy if total_buy else 0
summary["淨買超占比"] = summary["淨買超"] / total_net if total_net else 0

# 5. 排序與顯示（依淨買超大至小）
summary = summary.sort_values("淨買超", ascending=False)
summary.reset_index(drop=True, inplace=True)

summary

,券商,買進股數,賣出股數,淨買超,買進占比,淨買超占比
0,9800,139550016.0,75970482.0,63579534.0,0.272566,0
1,9300,34732785.0,19266314.0,15466471.0,0.067839,0
2,8150,13434925.0,828378.0,12606547.0,0.026241,0
3,9359,9249430.0,133210.0,9116220.0,0.018066,0
4,7000,6852797.0,647329.0,6205468.0,0.013385,0
...,...,...,...,...,...,...
806,9200,13251871.0,19344000.0,-6092129.0,0.025883,0
807,5920,11914427.0,22773565.0,-10859138.0,0.023271,0
808,1650,33123505.0,50320982.0,-17197477.0,0.064696,0
809,9600,9931115.0,30685930.0,-20754815.0,0.019397,0


,價格,券商,日期,買進股數,賣出股數,淨買超
69680,85.2,1020,2025-09-01,1000.0,0.0,1000.0
69681,87.2,1020,2025-09-01,3000.0,0.0,3000.0
69682,87.6,1020,2025-09-01,0.0,1000.0,-1000.0
69683,85.5,1021,2025-09-01,1000.0,0.0,1000.0
69684,85.6,1021,2025-09-01,102.0,0.0,102.0
...,...,...,...,...,...,...
77167,87.0,9A9X,2025-09-12,1000.0,0.0,1000.0
77168,87.2,9A9X,2025-09-12,0.0,1000.0,-1000.0
77169,87.3,9A9X,2025-09-12,1000.0,0.0,1000.0
77170,87.5,9A9X,2025-09-12,0.0,5000.0,-5000.0


In [79]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

HTML_PARSER = 'html.parser'

start_date = '11404'
end_date = '11410'
market_type_list = ['1', '2'] # 1: 上市, 2: 上櫃
expired_flags = ['0', '1'] # 0: 已到期, 1: 未到期 
for market_type in market_type_list:
    for expired in expired_flags:
        resp = requests.post(f'https://mopsov.twse.com.tw/mops/web/ajax_t90sb01?encodeURIComponent=1&step=1&firstin=1&off=1&r={market_type}&rc={expired}&start_date={start_date}&end_date={end_date}')
        soup = BeautifulSoup(resp.text, HTML_PARSER)
        filename = ''
        for element in soup.find_all('input', type='hidden'):
            if element.get('name') == 'filename':
                filename = element.get('value')
                break
        download_csv_url = f'https://mopsov.twse.com.tw/server-java/t105sb02?firstin=true&step=10&filename={filename}'
        csv_resp = requests.post(download_csv_url)
        break
    break

In [91]:
result[-1]

'"089999e","慧洋群益44購02","群益金鼎證券股份有限公司","認購","主動報價","2024/04/19",2024/04/19,2025/04/16,2025/04/18,"5",10000,"2637","慧洋-KY",291.00,78.00,"--","--",74.98,"--","--"'

In [105]:
import logging

In [114]:
parquet_path = '~/Desktop/bs_report/parquet_twse'
warrant_info_path = '~/StockAnalysis/data/warrant_list_dedup.csv'
try:
    df = pd.read_csv(warrant_info_path, dtype=str, encoding="utf-8-sig")
except Exception as exc:  # pragma: no cover
    logging.warning("Failed to read warrant list %s: %s", warrant_info_path, exc)

df = df.replace({np.nan: ""})
if "權證代號" not in df.columns or "標的代號" not in df.columns:
    logging.warning("Warrant list missing required columns: %s", df.columns.tolist())

df["權證代號"] = df["權證代號"].astype(str).str.strip()
df["標的代號"] = df["標的代號"].astype(str).str.strip()
df = df[df["權證代號"].str.len() > 4]
_warrant_lookup = df.copy()

In [117]:
underlying = 1216
if _warrant_lookup is None or _warrant_lookup.empty:
    logging.warning('_warrant_lookup is empty')
df = _warrant_lookup
mask = df["標的代號"].astype(str).str.strip() == str(underlying)
subset = df.loc[mask].copy()
if subset.empty:
    logging.warning('subset is empty')
subset["權證代號"] = subset["權證代號"].astype(str).str.strip()
subset = subset[subset["權證代號"].str.len() > 4]
subset.drop_duplicates("權證代號", inplace=True)
subset

,權證代號,權證簡稱,發行機構名稱,權證類型,流動量提供者報價方式,上市日期,履約開始日,最後交易日,履約截止日,結算方式說明(詳備註一),...,最新標的履約配發數量(每仟單位權證),原始履約價格(元)/履約點數(備註二),原始上限價格(元)/上限點數,原始下限價格(元)/下限點數,最新履約價格(元)/履約點數(備註二),最新上限價格(元)/上限點數,最新下限價格(元)/下限點數,market_type,expired_flag,上櫃日期
4334,049147,統一國泰44購01,國泰綜合證券股份有限公司,認購,主動報價,2024/08/30,2024/08/30,2025/04/25,2025/04/29,5,...,341.0,92.0,--,--,92.0,--,--,1,0,
4360,049179,統一國票45購01,國票綜合證券股份有限公司,認購,主動報價,2024/09/02,2024/09/02,2025/04/29,2025/05/02,5,...,250.0,105.0,--,--,105.0,--,--,1,0,
4510,049392,統一元大46購01,元大證券股份有限公司,認購,主動報價,2024/09/03,2024/09/03,2025/05/28,2025/06/02,5,...,250.0,97.5,--,--,97.5,--,--,1,0,
13452,059241,統一凱基45購01,凱基證券股份有限公司,認購,主動報價,2024/11/26,2024/11/26,2025/05/22,2025/05/26,5,...,250.0,100.0,--,--,100.0,--,--,1,0,
13512,059304,統一群益45購01,群益金鼎證券股份有限公司,認購,主動報價,2024/11/26,2024/11/26,2025/05/22,2025/05/26,5,...,300.0,100.0,--,--,100.0,--,--,1,0,
13823,059642,統一第一47購01,第一金證券股份有限公司,認購,主動報價,2024/11/28,2024/11/28,2025/07/24,2025/07/28,5,...,170.0,96.32,--,--,96.32,--,--,1,0,
17030,062850,統一永豐47購01,永豐金證券股份有限公司,認購,主動報價,2024/12/19,2024/12/19,2025/07/16,2025/07/18,5,...,168.0,90.8,--,--,90.8,--,--,1,0,
17885,063732,統一國票46購01,國票綜合證券股份有限公司,認購,主動報價,2024/12/25,2024/12/25,2025/06/20,2025/06/24,5,...,250.0,85.0,--,--,85.0,--,--,1,0,
19720,065884,統一凱基47購01,凱基證券股份有限公司,認購,主動報價,2025/01/08,2025/01/08,2025/07/03,2025/07/07,5,...,300.0,90.0,--,--,90.0,--,--,1,0,
20380,066711,統一群益47購01,群益金鼎證券股份有限公司,認購,主動報價,2025/01/13,2025/01/13,2025/07/10,2025/07/14,5,...,200.0,85.0,--,--,85.0,--,--,1,0,


https://www.tpex.org.tw/www/zh-tw/afterTrading/otc?date=2025%2F10%2F30&type=AL&id=&response=json


In [156]:
pdf.shape

(12166, 17)

In [ ]:
證券代號,證券名稱,成交股數,成交筆數,成交金額,開盤價,最高價,最低價,收盤價,漲跌(+/-),漲跌價差,最後揭示買價,最後揭示買量,最後揭示賣價,最後揭示賣量,本益比
